In [ ]:
# %% [markdown]
# # 03 — Demand Forecasting (Prophet + LSTM Ensemble)
# ## RetailPulse — Zidio Development | March 2026


In [ ]:
# %%[markdown]
# ## Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from prophet import Prophet
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150

In [ ]:
# %%[markdown]
# ## Load Daily Sales Data

In [ ]:
daily_sales = pd.read_parquet('../data/processed/daily_sales_ts.parquet')
print(f"Rows: {len(daily_sales):,}")
# Aggregate to total daily revenue for time series
ts = daily_sales.groupby('date')['total_revenue'].sum().reset_index()
ts.columns = ['ds', 'y']
ts['ds'] = pd.to_datetime(ts['ds'])
ts = ts.sort_values('ds').reset_index(drop=True)

# Fill missing dates
date_range = pd.date_range(ts['ds'].min(), ts['ds'].max(), freq='D')
ts = ts.set_index('ds').reindex(date_range).fillna(0).reset_index()
ts.columns = ['ds', 'y']

print(f"Date range: {ts['ds'].min()} to {ts['ds'].max()}")
print(f"Total days: {len(ts)}")
print(f"Total revenue: £{ts['y'].sum():,.0f}")

In [ ]:
# %%[markdown]
# ## Stationarity Test (ADF)

In [ ]:
adf_result = adfuller(ts['y'].dropna())
print(f"ADF Statistic: {adf_result[0]:.4f}")
print(f"p-value: {adf_result[1]:.4f}")
print(f"Stationary: {adf_result[1] < 0.05}")
print(f"Critical values: {adf_result[4]}")

In [ ]:
# %%[markdown]
# ## Seasonal Decomposition

In [ ]:
decomp = seasonal_decompose(ts.set_index('ds')['y'], model='additive', period=7)
fig = decomp.plot()
fig.set_size_inches(12, 10)
plt.suptitle('Seasonal Decomposition (7-day period)', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/seasonal_decomposition.png', bbox_inches='tight')
plt.show()

In [ ]:
# %%[markdown]
# ## Prophet Model — Baseline

In [ ]:
train = ts[ts['ds'] < '2010-10-01']
test = ts[ts['ds'] >= '2010-10-01']
print(f"Train: {len(train)} days, Test: {len(test)} days")

model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    seasonality_mode='multiplicative'
)
model.add_country_holidays(country_name='GB')
model.fit(train)

future = model.make_future_dataframe(periods=30)
forecast = model.predict(future)

In [ ]:
# %%[markdown]
# ## Evaluate

In [ ]:
merged = test.merge(forecast[['ds', 'yhat']], on='ds', how='inner')
mape = (abs(merged['y'] - merged['yhat']) / (merged['y'] + 1e-6)).mean() * 100
print(f"Prophet MAPE: {mape:.2f}%")
print(f"Within target (≤12%): {mape <= 12}")

fig1 = model.plot(forecast)
plt.title(f'Prophet Forecast (MAPE: {mape:.2f}%)')
plt.savefig('../reports/prophet_forecast.png', bbox_inches='tight')
plt.show()

fig2 = model.plot_components(forecast)
plt.savefig('../reports/prophet_components.png', bbox_inches='tight')
plt.show()

In [ ]:
# %%[markdown]
# ## Save Forecast Results

In [ ]:
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].to_csv(
    '../data/processed/forecast_results.csv', index=False)
print(f"Saved forecast_results.csv with {len(forecast)} rows")

# Save for ensemble
ensemble = forecast[['ds', 'yhat']].copy()
ensemble['model'] = 'prophet'
ensemble['mape'] = mape
ensemble.to_csv('../data/processed/ensemble_forecast_results.csv', index=False)
print("Saved ensemble_forecast_results.csv")